# Лабораторная №5. Классы и объекты

**Фокус Python:** self, __init__, методы, dataclass, property, композиция, наследование, ABC, dunder, classmethod.

**Подготовка:** предыдущие работы; их готовый код переносить не требуется. Python 3.11+. Обязательная практика использует стандартную библиотеку.
**Объём:** шесть небольших заданий; ориентир 90–150 минут с изучением примеров. Время ориентировочное, корректируется после первого проведения.

## Порядок работы

1. Прочитайте объяснение и **предскажите вывод примера до запуска**.
2. Запустите пример, сравните с прогнозом.
3. Реализуйте задание в своей ячейке; заготовки намеренно не завершены.
4. Запустите самопроверку и попробуйте иной вход. Не подставляйте ожидаемые ответы вместо вычислений.
5. Ответьте себе на короткий вопрос после задания. На защите будет похожая проверка и небольшое изменение кода.

Открытые проверки показывают маленькие примеры контракта и помогают учиться. Они не проверяют все случаи и не заменяют вашу реализацию. До сдачи добавьте **два собственных граничных случая**. Для работ 1–2 временно замените входы в рабочей ячейке, запустите только её и проверьте результат вручную; фиксированные самопроверки относятся к исходным данным, после эксперимента восстановите их.

Документация и помощь при подготовке допустимы. Любую заимствованную конструкцию нужно суметь объяснить и изменить самостоятельно. Слишком сложный непонятный код замените понятным. На защите проверяется понимание, а не память о точных числах.

**Оценка:** практика 12 баллов (6 × 2), объяснение 2, прогноз результата 2, небольшое изменение 4. Подробную рубрику объявляет преподаватель до сдачи.


In [57]:
from dataclasses import dataclass
from abc import ABC, abstractmethod

## 1. Состояние экземпляра

Класс описывает объекты. self — конкретный экземпляр. self.items в __init__ создаёт отдельный список; изменяемый атрибут класса разделяется экземплярами. Метод меняет состояние своего объекта.

**Перед запуском:** что выведет следующий пример и какого типа будут значения?

In [58]:
class _DemoBox:
    def __init__(self):
        self.items = []
_demo_a, _demo_b = _DemoBox(), _DemoBox()
_demo_a.items.append("x")
print(_demo_a.items, _demo_b.items)

['x'] []


### Самостоятельно — 2 балла

Создайте CounterBox(start=0), храните value. inc() увеличивает value на 1 и возвращает новое значение, reset() устанавливает 0 и возвращает None. start — корректное целое.

Реализуйте только описанный контракт. Проверка произвольных других типов не нужна, если она не запрошена явно.

In [59]:
class CounterBox:
    def __init__(self, start=0):
        self.value = start

    def inc(self):
        self.value += 1
        return self.value

    def reset(self):
        self.value = 0

**Самопроверка.** Сначала реализуйте код выше. Ошибка в проверке — повод найти несоответствие контракту, а не удалить `assert`.

In [60]:
a, b = CounterBox(2), CounterBox()
assert a.inc() == 3 and b.value == 0
assert a.reset() is None and a.value == 0

**Проверьте понимание:** Почему изменение a не должно менять b?
Потому что это разные объекты
Ответьте устно или одной фразой; отдельный длинный отчёт не нужен.

## 2. Модель заметки

dataclass создаёт __init__, __repr__, сравнение по полям. __post_init__ вызывается после созданного инициализатора. property позволяет вычислять атрибут при чтении; его не нужно дублировать в состоянии.

**Перед запуском:** что выведет следующий пример и какого типа будут значения?

In [61]:
@dataclass
class _DemoLabel:
    text: str
    @property
    def size(self):
        return len(self.text)
print(_DemoLabel("abc").size)

3


### Самостоятельно — 2 балла

Note — dataclass с title: str, done: bool=False. __post_init__ очищает title через strip и запрещает пустое имя (ValueError). Типы входа корректны. complete() ставит done=True, возвращает None. property label возвращает "[x] имя" для выполненной, иначе "[ ] имя".

Реализуйте только описанный контракт. Проверка произвольных других типов не нужна, если она не запрошена явно.

In [62]:
@dataclass
class Note:
    title: str
    done: bool = False

    def __post_init__(self):
        self.title = self.title.strip()
        if self.title == '':
            raise ValueError('Имя не должно быть пустым')

    def complete(self):
        self.done = True

    @property
    def label(self):
        if self.done == True:
            return f'[x] {self.title}'
        else:
            return f'[ ] {self.title}'

**Самопроверка.** Сначала реализуйте код выше. Ошибка в проверке — повод найти несоответствие контракту, а не удалить `assert`.

In [63]:
note = Note(" Читать ")
assert note.label == "[ ] Читать"
assert note.complete() is None and note.label == "[x] Читать"
try:
    Note(" ")
except ValueError:
    pass
else:
    raise AssertionError("Пустое имя разрешено")

**Проверьте понимание:** Почему label лучше вычислять, а не хранить отдельной строкой?

Ответьте устно или одной фразой; отдельный длинный отчёт не нужен.

## 3. Композиция и протоколы

Композиция: блокнот содержит заметки. __len__ обслуживает len(), __getitem__ — индекс, __contains__ — in. __iter__ возвращает итератор; подробно он будет разобран в работе 6. Внутренний список можно скопировать, оставив сами заметки общими.

**Перед запуском:** что выведет следующий пример и какого типа будут значения?

In [64]:
class _DemoShelf:
    def __init__(self, items):
        self.items = list(items)
    def __len__(self):
        return len(self.items)
print(len(_DemoShelf([1, 2])))

2


### Самостоятельно — 2 балла

Notebook(notes=None) копирует внешний список заметок. add(note) добавляет объект; len — число; индекс — Note; in ищет точное title; __iter__ возвращает iter внутреннего списка; repr строго "Notebook(notes=N)". Пустой блокнот допустим.

Реализуйте только описанный контракт. Проверка произвольных других типов не нужна, если она не запрошена явно.

In [83]:
class Notebook:
    def __init__(self, notes=None):
        self.notes = list(notes)

    def add(self, note):
        self.notes.append(note)

    def __len__(self):
        return len(self.notes)
    
    def __getitem__(self, index):
        return self.notes[index]

    def __contains__(self, title):
        for note in self.notes:
            if note.title == title:
                return True
        return False
        

    def __iter__(self):
        return iter(self.notes)

    def __repr__(self):
        return f'Notebook(notes={len(self.notes)})'

**Самопроверка.** Сначала реализуйте код выше. Ошибка в проверке — повод найти несоответствие контракту, а не удалить `assert`.

In [84]:
source = [Note("A")]
book = Notebook(source)
source.append(Note("B"))
print(len(book))
assert len(book) == 1 and book[0] is source[0]
assert "A" in book and "B" not in book
assert list(book) == [source[0]] and repr(book) == "Notebook(notes=1)"

1


**Проверьте понимание:** Какой уровень копирования гарантирован?

Ответьте устно или одной фразой; отдельный длинный отчёт не нужен.

## 4. Общий интерфейс

Наследование задаёт разновидность, композиция — содержимое. ABC и abstractmethod запрещают создавать объект с незавершённым интерфейсом. Полиморфизм: вызывающий код использует один метод без проверки конкретного типа.

**Перед запуском:** что выведет следующий пример и какого типа будут значения?

In [85]:
class _DemoSpeaker(ABC):
    @abstractmethod
    def speak(self):
        pass
class _DemoCat(_DemoSpeaker):
    def speak(self):
        return "мяу"
print(_DemoCat().speak())

мяу


### Самостоятельно — 2 балла

Создайте Formatter(ABC) с abstractmethod format(note). PlainFormatter возвращает note.title, StatusFormatter — note.label. render_notes(notes, formatter) возвращает список результатов format, не использует isinstance для выбора форматирования.

Реализуйте только описанный контракт. Проверка произвольных других типов не нужна, если она не запрошена явно.

In [86]:
class Formatter(ABC):
    @abstractmethod
    def format(self, note):
        pass

class PlainFormatter(Formatter):
    def format(self, note):
        return note.title

class StatusFormatter(Formatter):
    def format(self, note):
        return note.label

def render_notes(notes, formatter):
    return [formatter.format(note) for note in notes]

**Самопроверка.** Сначала реализуйте код выше. Ошибка в проверке — повод найти несоответствие контракту, а не удалить `assert`.

In [87]:
notes = [Note("A"), Note("B", True)]
assert render_notes(notes, PlainFormatter()) == ["A", "B"]
assert render_notes(notes, StatusFormatter()) == ["[ ] A", "[x] B"]

**Проверьте понимание:** Нужно ли менять render_notes при добавлении нового Formatter?

Ответьте устно или одной фразой; отдельный длинный отчёт не нужен.

## 5. Альтернативный конструктор

classmethod получает класс cls, а не экземпляр self. Он может создать экземпляр из другого представления. Использование cls(...) вместо жёсткого имени класса поддерживает наследование. Данные перед созданием нужно проверить существующей моделью.

**Перед запуском:** что выведет следующий пример и какого типа будут значения?

In [90]:
class _DemoWord:
    def __init__(self, text):
        self.text = text
    @classmethod
    def from_number(cls, value):
        return cls(str(value))
print(_DemoWord.from_number(12).text)

12


### Самостоятельно — 2 балла

Создайте ImportableNotebook(Notebook) с classmethod from_titles(titles). Каждый элемент передайте в Note, затем соберите cls из заметок. Пустой список допустим; ошибка пустого title распространяется.

Реализуйте только описанный контракт. Проверка произвольных других типов не нужна, если она не запрошена явно.

In [91]:
class ImportableNotebook(Notebook):
    @classmethod
    def from_titles(cls, titles):
        notes = [Note(title) for title in titles]
        return cls(notes)

**Самопроверка.** Сначала реализуйте код выше. Ошибка в проверке — повод найти несоответствие контракту, а не удалить `assert`.

In [92]:
imported = ImportableNotebook.from_titles([" A ", "B"])
assert isinstance(imported, ImportableNotebook)
assert [n.title for n in imported] == ["A", "B"]
assert len(ImportableNotebook.from_titles([])) == 0

**Проверьте понимание:** Чем cls отличается от self?

Ответьте устно или одной фразой; отдельный длинный отчёт не нужен.

## 6. Объекты и обычные данные

JSON не знает произвольный класс Note. Явный словарь отделяет состояние объекта от способов его хранения. При восстановлении вызывайте конструктор: это повторно применит проверку инвариантов.

**Перед запуском:** что выведет следующий пример и какого типа будут значения?

In [93]:
_demo_note = {"title": "Read", "done": False}
print(_demo_note["title"])

Read


### Самостоятельно — 2 балла

notebook_to_data(book) возвращает список {title, done}. notebook_from_data(data) создаёт Notebook из Note. Входная структура и типы корректны; проверка title остаётся в Note. Возвращённые словари новые: их изменение не должно менять заметки.

Реализуйте только описанный контракт. Проверка произвольных других типов не нужна, если она не запрошена явно.

In [96]:
def notebook_to_data(book):
    return [{'title': note.title, 'done': note.done} for note in book]

def notebook_from_data(data):
    notes = [Note(item['title'], item['done']) for item in data]
    return Notebook(notes)

**Самопроверка.** Сначала реализуйте код выше. Ошибка в проверке — повод найти несоответствие контракту, а не удалить `assert`.

In [97]:
book = Notebook([Note("A", True)])
data = notebook_to_data(book)
restored = notebook_from_data(data)
assert notebook_to_data(restored) == data
data[0]["title"] = "changed"
assert book[0].title == "A" and restored[0].title == "A"

**Проверьте понимание:** Почему нельзя просто вернуть внутренние объекты как JSON?

Ответьте устно или одной фразой; отдельный длинный отчёт не нужен.

## Перед сдачей

- Реализованы шесть заданий; выполняются с чистого ядра.
- Добавлены два собственных граничных случая и объяснено, что они проверяют.
- Вы можете назвать тип результата, объяснить одну важную строку и изменить условие.
- Необязательное усложнение: придумайте одно расширение контракта и тест к нему. Оно не требуется для максимальной оценки.

### Три коротких наблюдения

1. Я ожидал(а) …, а получилось …, потому что …
2. Я исправил(а) ошибку … и проверил(а) это на …
3. Я пока путаю … / могу объяснить …

Заполните этот Markdown-блок своими словами. На защите преподаватель выберет один небольшой участок, а не попросит пересказывать всю работу.
